# Tool calling — and the description as prompt

**Session 6 · Track B · hosted OpenAI**

Give the model a tool; measure how the description changes tool choice.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
# set MODEL_BACKEND=openai in .env for this lab
from openai import OpenAI


### Worked example

One typed tool, a good description vs a broken one, and tool-selection accuracy over prompts that should and should not trigger it.

_Needs `MODEL_BACKEND=openai` and `OPENAI_API_KEY` in `.env`._


In [ ]:
# Worked example: the description IS the prompt - measure it
import os
from openai import OpenAI

client = OpenAI()
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

def tool_spec(description):
    return [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": description,
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name"}},
                "required": ["city"],
            },
        },
    }]

GOOD   = ("Get the current weather for a city. Use whenever the user asks about "
          "weather, temperature, rain, or the forecast.")
BROKEN = "Does stuff."

SHOULD_CALL = ["What's the weather in Paris?", "Will it rain in Tokyo tomorrow?", "temperature in Cairo?"]
SHOULD_NOT  = ["Who painted the Mona Lisa?", "Translate 'hello' into Spanish."]

def selects_tool(prompt, description):
    r = client.chat.completions.create(
        model=MODEL, tools=tool_spec(description),
        messages=[{"role": "user", "content": prompt}],
    )
    return bool(r.choices[0].message.tool_calls)

for name, desc in [("GOOD", GOOD), ("BROKEN", BROKEN)]:
    hits = sum(selects_tool(p, desc) for p in SHOULD_CALL)
    miss = sum(selects_tool(p, desc) for p in SHOULD_NOT)
    total = len(SHOULD_CALL) + len(SHOULD_NOT)
    print(f"{name:7} selection accuracy: {hits + (len(SHOULD_NOT) - miss)}/{total}  "
          f"(fired {hits}/{len(SHOULD_CALL)} wanted, {miss}/{len(SHOULD_NOT)} unwanted)")


## Your turn - vary the example

1. Add a second tool (e.g. `currency_convert`) and prompts that should route to each.
2. Degrade the GOOD description one word at a time - where does routing break?
3. Add an ambiguous prompt ("how's Paris?"). Which tool fires, and is that right?


In [ ]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
